# NumPy Lab: Slicing, Advanced Indexing, Reshaping & Broadcasting

**Duration:** ~60 minutes  
**Last updated:** 2025-09-28 01:44

### Learning goals
1. Distinguish **views** vs **copies** when slicing ndarrays.
2. Apply **advanced indexing** (boolean masks & fancy indexing).
3. Use **reshape** and **broadcasting** for vectorized operations.

> **Data:** This lab uses `SampleData.csv` with columns like `Account_no`, `Billed_usage_kwh`, `Invoice_date`, `Price`, `Bill`, etc.

## 0) Setup & Data Load

- If running here, the CSV is at `/mnt/data/SampleData.csv`.
- If running elsewhere, place `SampleData.csv` next to your notebook and update `PATH`.

We'll create:
- `df` – pandas DataFrame
- `A` – NumPy array with numeric columns `[Billed_usage_kwh, Base_charge, Price, Bill]`
- helpers: `acct`, `month`, `product`

In [ ]:
import numpy as np
import pandas as pd

PATH = r"SampleData.csv"  # change to "SampleData.csv" if running locally
df = pd.read_csv(PATH, parse_dates=["Invoice_date"], dtype={"Account_no": "string"})

# Parse date for month-based tasks
df["Invoice_date"] = pd.to_datetime(df["Invoice_date"], format="%m/%d/%Y")

num_cols = ["Billed_usage_kwh", "Base_charge", "Price", "Bill"]
A = df[num_cols].to_numpy(dtype=np.float64)  # shape (N,4)

acct = df["Account_no"].to_numpy()
month = df["Invoice_date"].dt.month.to_numpy()
product = df["ProductID"].to_numpy()

print("Shape of df:", df.shape)
print("A shape:", A.shape)
print(A)
df.head(3)

## Part 1 — Slicing, Views & Copies

**Goal:** Understand when a slice is a view (mutations reflect back) vs when you have a real copy.

### 1.1 Basic slicing (views)

In [ ]:
print("A", A)

S1 = A[:10, :2]
S2 = A[::2, :]
S3 = A[-5:, -1]

print("S1", S1)
print("S2", S2)
print("S3", S3)

print("S1", S1.shape, "shares?", np.shares_memory(A, S1))
print("S2", S2.shape, "shares?", np.shares_memory(A, S2))
print("S3", S3.shape, "shares?", np.shares_memory(A, S3))

### 1.2 Prove a slice is a view (mutations propagate)

In [ ]:
A_copy = A.copy()
sub = A_copy[:5, 0]   # first 5 rows of usage
sub += 1000
print("Mutated first 6 usage values:", A_copy[:6, 0])

### 1.3 Real copy to prevent side effects

In [ ]:
A_safe = A.copy()
sub2 = A_safe[:5, 0].copy()
sub2 += 1000
print("Check first 6 usage values (should be unchanged):", A_safe[:6, 0])

### 1.4 Contiguity & ravel vs flatten

In [ ]:
print("A C-contiguous?", A.flags['C_CONTIGUOUS'])
r = A.ravel()
f = A.flatten()
print("shares_memory(A, ravel)?", np.shares_memory(A, r))
print("shares_memory(A, flatten)?", np.shares_memory(A, f))

## Part 2 — Advanced Indexing & Selection

**Goal:** Use boolean masks, combined criteria, fancy indexing, and `np.where` for conditional logic.

### 2.1 Boolean masks (filter rows by condition)

In [ ]:
mask_hi_usage = A[:, 0] > 1000  # usage > 1000 kWh
hi_usage_rows = A[mask_hi_usage]
# print(hi_usage_rows)
hi_usage_rows.shape, mask_hi_usage.sum()

### 2.2 Combine conditions (AND / OR)

In [ ]:
mask_july = month == 7
mask_price = A[:, 2] >= 0.03
mask_combo = mask_hi_usage & mask_july & mask_price

sel = A[mask_combo]
sel.shape, mask_combo.sum()

### 2.3 In-place assignment with boolean masks

In [ ]:
B = A.copy()
mask_discount = B[:, 0] > 1200
B[mask_discount, 3] *= 0.9
(B[mask_discount, 3] <= A[mask_discount, 3]).all()

### 2.4 Fancy indexing with integer arrays (copy semantics)

In [ ]:
idx = np.array([0, 2, 4, 6, 8])
sub_int = A[idx, :]
# print(sub_int)
print("shares_memory(A, sub_int)?", np.shares_memory(A, sub_int))
sub_int[:, 3] += 1.23
# print(sub_int)
(A[idx, 3] == sub_int[:, 3]).all()  # Expect False

### 2.5 `np.where` for vectorized conditional logic

In [ ]:
usage = A[:, 0]
bill = A[:, 3]
unit_cost = np.where(usage != 0, bill / usage, np.nan)
unit_cost[:10]

## Part 3 — Reshaping & Broadcasting

**Goal:** Apply `keepdims`, `reshape`, `np.newaxis`, and broadcasting rules for clean vectorization.

### 3.1 Column centering via broadcasting

In [ ]:
means = A.mean(axis=0, keepdims=True)    # (1,4)
print("A", A)
print("A.shape", A.shape)
print("means", means)
print("means.shape", means.shape)
A_centered = A - means
np.allclose(A_centered.mean(axis=0), 0.0, atol=1e-10)

### 3.2 Z-score standardization

In [ ]:
stds = A.std(axis=0, ddof=0, keepdims=True)  # (1,4)
Z = (A - means) / stds
np.allclose(Z.mean(axis=0), 0.0, atol=1e-10), np.allclose(Z.std(axis=0), 1.0, atol=1e-8)

### 3.3 Reshape vectors for broadcasting

In [ ]:
m_col = month.reshape(-1, 1)          # (N,1)
w_row = np.arange(1, 5).reshape(1, -1)  # (1,4)
weighted = A * (m_col / 12) * w_row
weighted.shape

### 3.4 Row-wise normalization using broadcasting

In [ ]:
row_den = A[:, 3:4]   # Bill as (N,1)
norm_by_bill = np.where(row_den != 0, A / row_den, 0.0)
norm_by_bill.shape

### 3.5 Month-to-month bill deltas (global)

In [ ]:
bill = A[:, 3]
bill_delta = bill[1:] - bill[:-1]
bill_delta[:10]

## Stretch Goals (with Solutions)

### S1) Per-account month-to-month usage deltas (NumPy) — **Solution**

In [ ]:
# Ensure sorted by (account, date) so diffs apply per-account in order
order = np.lexsort((df["Invoice_date"].to_numpy(), acct))
A_sorted = A[order]
acct_sorted = acct[order]
usage_sorted = A_sorted[:, 0]

usage_delta = np.full(usage_sorted.shape, np.nan, dtype=float)
boundaries = np.flatnonzero(acct_sorted[1:] != acct_sorted[:-1]) + 1
starts = np.r_[0, boundaries]
ends   = np.r_[boundaries, usage_sorted.shape[0]]

for s, e in zip(starts, ends):
    if e - s >= 2:
        usage_delta[s+1:e] = np.diff(usage_sorted[s:e])

# (Optional) Put back to original order
inv = np.empty_like(order)
inv[order] = np.arange(order.size)
usage_delta_original_order = usage_delta[inv]

# Show first 10 deltas alongside account numbers
np.column_stack((acct[:10], usage_delta_original_order[:10]))

### S2) “High bill in summer” mask & stats — **Solution**

In [ ]:
usage = A[:, 0]
bill  = A[:, 3]

summer_mask = (month == 6) | (month == 7) | (month == 8)
high_bill_mask = bill > 30.0
mask = summer_mask & high_bill_mask

count_high_summer = mask.sum()
unit_cost = np.where(usage != 0, bill / usage, np.nan)
avg_usage_high_summer = usage[mask].mean() if count_high_summer else np.nan
avg_unit_cost_high_summer = unit_cost[mask].mean() if count_high_summer else np.nan

count_high_summer, float(avg_usage_high_summer), float(avg_unit_cost_high_summer)

### S3) Pairwise absolute bill differences — **Solution**

In [ ]:
k = 200  # adjust based on memory
b = A[:k, 3]  # bill
pair_diff = np.abs(b[:, None] - b[None, :])
pair_diff.shape, float(pair_diff.mean()), float(pair_diff.max())

### S4) Pitfall: slice(view) vs fancy(copy) — **Solution**

In [ ]:
# Slice -> view
S = A[:5, :2]
print("Slice shares memory?", np.shares_memory(A, S))
S[:, 0] += 999
print("A changed via slice mutation?", np.allclose(A[:5, 0], S[:, 0]))
S[:, 0] -= 999  # undo

# Fancy indexing -> copy
idx = np.array([0,1,2,3,4])
F = A[idx, :2]
print("Fancy shares memory?", np.shares_memory(A, F))
F[:, 0] += 777
print("A unchanged after fancy mutation?", not np.allclose(A[idx, 0], F[:, 0]))

## Reflection
- When do you get **views** vs **copies**?
- Why is **broadcasting** powerful for vectorization?
- When do you need `reshape`, `None/np.newaxis`, or `keepdims=True`?